# 00 — Environment Setup & Reproducibility Verification

**Project:** TURBINE-X — Gas Turbine Emission Prediction and ML Benchmarking System  
**Objective:** Verify Python interpreter, hardware architecture, dependency matrix, directory discovery, random seeds, and logging infrastructure.  
**Hardware Target:** CPU-First Benchmarking (Windows, AMD Ryzen 9 / Intel, 16GB RAM)


In [1]:
import os
import sys
import platform
import psutil
import json
import logging
from pathlib import Path

# Locate project root dynamically
def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
print(f"Discovered Project Root: {ROOT}")
sys.path.insert(0, str(ROOT))


Discovered Project Root: D:\TURBINE_X_Full_Project\TURBINE_X_Full_Project


## 1. System Hardware & Python Environment Audit
Checking operating system, CPU architecture, RAM, and Python version compatibility.


In [2]:
env_info = {
    "os": platform.system(),
    "os_release": platform.release(),
    "os_version": platform.version(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cpu_count_logical": psutil.cpu_count(logical=True),
    "cpu_count_physical": psutil.cpu_count(logical=False),
    "ram_total_gb": round(psutil.virtual_memory().total / (1024**3), 2),
    "python_version": platform.python_version(),
    "python_executable": sys.executable
}
for k, v in env_info.items():
    print(f"  {k:22s}: {v}")


  os                    : Windows
  os_release            : 11
  os_version            : 10.0.26200
  machine               : AMD64
  processor             : AMD64 Family 25 Model 97 Stepping 2, AuthenticAMD
  cpu_count_logical     : 32
  cpu_count_physical    : 16
  ram_total_gb          : 15.21
  python_version        : 3.13.14
  python_executable     : D:\TURBINE_X_Full_Project\.venv\Scripts\python.exe


## 2. Core Dependency Verification
Verifying that all required numerical computing, machine learning, optimization, and visualization packages are installed.


In [3]:
packages = [
    "numpy", "pandas", "scipy", "sklearn", "matplotlib", 
    "seaborn", "plotly", "joblib", "yaml", "xgboost", 
    "catboost", "optuna", "flaml", "shap", "streamlit", "pytest"
]

installed_deps = {}
for pkg in packages:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "installed")
        installed_deps[pkg] = ver
        print(f"  [OK] {pkg:15s} : {ver}")
    except ImportError as e:
        installed_deps[pkg] = f"MISSING: {e}"
        print(f"  [MISSING] {pkg:15s} : {e}")


  [OK] numpy           : 2.5.4


  [OK] pandas          : 2.3.3
  [OK] scipy           : 1.18.1


  [OK] sklearn         : 1.9.1


  [OK] matplotlib      : 3.11.2


  [OK] seaborn         : 0.13.2
  [OK] plotly          : 7.1.0
  [OK] joblib          : 1.6.0
  [OK] yaml            : 6.0.3


  [OK] xgboost         : 2.1.4


  [OK] catboost        : 1.2.10
  [OK] optuna          : 5.0.0


D:\TURBINE_X_Full_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  [OK] flaml           : 2.7.0


  [OK] shap            : 0.54.1


  [OK] streamlit       : 1.66.1
  [OK] pytest          : 9.2.0


## 3. Directory Layout and Raw Dataset Verification
Ensuring standard directory hierarchy (`data/`, `configs/`, `outputs/`, `experiments/`, `models/`, `artifacts/`) exists.


In [4]:
dirs_to_verify = [
    ROOT / "configs",
    ROOT / "data" / "raw",
    ROOT / "data" / "processed",
    ROOT / "outputs" / "figures",
    ROOT / "outputs" / "reports",
    ROOT / "outputs" / "tables",
    ROOT / "experiments",
    ROOT / "artifacts" / "models",
    ROOT / "models",
    ROOT / "logs"
]

for d in dirs_to_verify:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  Directory verified: {d.relative_to(ROOT)}")

# Verify presence of raw CSV files
raw_dir = ROOT / "data" / "raw"
raw_files = [f"gt_{yr}.csv" for yr in range(2011, 2016)]
for f in raw_files:
    csv_path = raw_dir / f
    assert csv_path.exists(), f"Missing dataset: {f}"
    print(f"  Found dataset: {f} ({csv_path.stat().st_size:,} bytes)")


  Directory verified: configs
  Directory verified: data\raw
  Directory verified: data\processed
  Directory verified: outputs\figures
  Directory verified: outputs\reports
  Directory verified: outputs\tables
  Directory verified: experiments
  Directory verified: artifacts\models
  Directory verified: models
  Directory verified: logs
  Found dataset: gt_2011.csv (562,921 bytes)
  Found dataset: gt_2012.csv (577,617 bytes)
  Found dataset: gt_2013.csv (540,334 bytes)
  Found dataset: gt_2014.csv (541,009 bytes)
  Found dataset: gt_2015.csv (556,928 bytes)


## 4. Logging & Environment Check Export
Configuring file logging and saving environment manifest for task registry validation.


In [5]:
log_file = ROOT / "logs" / "project.log"
logging.basicConfig(
    filename=str(log_file),
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
logging.info("TURBINE-X environment verification initialized successfully.")

# Save environment_check.json
env_report = {
    "environment": env_info,
    "dependencies": installed_deps,
    "project_root": str(ROOT),
    "status": "VERIFIED"
}

out_report = ROOT / "outputs" / "reports" / "environment_check.json"
out_report.parent.mkdir(parents=True, exist_ok=True)
with open(out_report, "w", encoding="utf-8") as f:
    json.dump(env_report, f, indent=2)

print(f"Saved environment manifest to: {out_report.relative_to(ROOT)}")
print("Environment and setup verification COMPLETE.")


Saved environment manifest to: outputs\reports\environment_check.json
Environment and setup verification COMPLETE.
